# Movies, Ratings & Oscars — Data Cleaning

**Research questions**
1. Does a bigger budget produce more revenue?
2. Do Oscar-winning movies receive higher ratings?
3. Which genres generate the highest revenue?
4. Does movie runtime influence ratings?
5. Which directors consistently produce successful movies?

**Data sources**
- `title_basics.tsv` (**movies_2**) — IMDb titles: `tconst`, `primaryTitle`, `startYear`, `runtimeMinutes`, `genres`.
- `title.ratings.tsv` (**ratings**) — IMDb ratings: `tconst`, `averageRating`, `numVotes`.
- `TMDB_movie_dataset_v11.csv` (**movies**) — budget, revenue, `imdb_id`. Used to fill values *not* present in the movies_2 + ratings join (mainly **budget** / **revenue**).
- `full_data.csv` — Oscar nominations/wins, tab-separated, `FilmId` links to IMDb ids (`tt...`).


Join key throughout: `tconst`  ↔  TMDB `imdb_id`  ↔  Oscar `FilmId`.

## 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np

os.makedirs("output", exist_ok=True)   # all CSVs saved here

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load Data and Extraction

**Movie list from title_basics**

In [2]:
# --- movies_2: IMDb title basics ---
basics = pd.read_csv(
    "title_basics.tsv", sep="\t", na_values="\\N", low_memory=False,
    usecols=["tconst", "titleType", "primaryTitle",
             "startYear", "runtimeMinutes", "genres"],
    dtype={"startYear": "str", "runtimeMinutes": "str"})
basics = basics[basics["titleType"] == "movie"].copy()   # keep films only
print("movies_2 (movie rows):", basics.shape)
basics.head(3)


movies_2 (movie rows): (750860, 6)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,1894,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905,100,NaN


In [3]:
basics.dtypes

tconst            object
titleType         object
primaryTitle      object
startYear         object
runtimeMinutes    object
genres            object
dtype: object

In [4]:
import csv
import unicodedata

# Make a copy
basics_new = basics.copy()

# Function to remove accents and quotes
def clean_text(v):
    if not isinstance(v, str):
        return v

    # Remove accents (é -> e, ü -> u, etc.)
    v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")

    # Remove double quotes
    v = v.replace('"', '')

    # Remove carriage returns/newlines inside fields
    v = v.replace("\r", " ").replace("\n", " ")

    return v.strip()

# Apply to every cell
basics_new = basics_new.map(clean_text)


# Export
basics_new.to_csv(
    "output/movies_base.csv",
    sep=",",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL
)

print("movies_base.csv:", basics_new.shape)

movies_base.csv: (750860, 6)


**Ratings: IMDb averageRating / numVotes**

In [5]:
# --- ratings: IMDb averageRating / numVotes ---
ratings = pd.read_csv("title.ratings.tsv", sep="\t", na_values="\\N")
print("ratings:", ratings.shape)
ratings.head(3)

ratings: (1690132, 3)


,tconst,averageRating,numVotes
0,tt0000001,5.70,2214
1,tt0000002,5.40,320
2,tt0000003,6.40,2351


In [6]:
ratings.averageRating.unique()

array([ 5.7,  5.4,  6.4,  5.1,  6.2,  5. ,  5.3,  6.8,  5.2,  7.4,  7.1,
        6.1,  5.9,  4.5,  4.7,  3.8,  4. ,  5.6,  5.5,  4.9,  4.4,  4.2,
        4.1,  6.7,  3.4,  3.5,  3.6,  4.8,  3.7,  3.1,  3.3,  6.3,  3.9,
        4.3,  4.6,  6. ,  5.8,  3.2,  6.5,  2.9,  2.6,  3. ,  2.8,  2.5,
        2.3,  2.1,  8.1,  7.2,  2.7,  7.3,  6.9,  6.6,  2.2,  1.9,  1.8,
        2. ,  1.7,  2.4,  7.9,  7.8,  7. ,  8.6,  7.6,  7.7,  8. ,  8.9,
        8.3,  9. ,  8.5,  9.1,  7.5,  8.4,  8.8,  8.2,  1.3,  8.7,  1. ,
        1.6,  1.1,  9.3,  9.4,  1.4,  1.5,  1.2,  9.2,  9.6,  9.5,  9.8,
        9.7,  9.9, 10. ])

In [7]:
ratings.averageRating.unique()
ratings["averageRating"].dtype

dtype('float64')

In [8]:
import csv
import unicodedata

# Make a copy
ratings_new = ratings.copy()

# Function to remove accents and quotes
def clean_text(v):
    if not isinstance(v, str):
        return v

    # Remove accents (é -> e, ü -> u, etc.)
    v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")

    # Remove double quotes
    v = v.replace('"', '')

    # Remove carriage returns/newlines inside fields
    v = v.replace("\r", " ").replace("\n", " ")

    return v.strip()

# Apply to every cell
ratings_new = ratings_new.map(clean_text)


# Export
ratings_new.to_csv(
    "output/ratings.csv",
    sep=",",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
    float_format="%.1f"
)

print("ratings.csv:", ratings_new.shape)

ratings.csv: (1690132, 3)


**Movies from TMDB**

In [9]:

movies = pd.read_csv("TMDB_movie_dataset_v11.csv")
print("movies (TMDB):", movies.shape)
movies[["imdb_id", "title", "budget", "revenue",
        "vote_average", "vote_count", "runtime", "genres"]].head(3)

movies (TMDB): (1451806, 24)


,imdb_id,title,budget,revenue,vote_average,vote_count,runtime,genres
0,tt1375666,Inception,160000000,825532764,8.36,34495,148,"Action, Science Fiction, Adventure"
1,tt0816692,Interstellar,165000000,701729206,8.42,32571,169,"Adventure, Drama, Science Fiction"
2,tt0468569,The Dark Knight,185000000,1004558444,8.51,30619,152,"Drama, Action, Crime, Thriller"


In [10]:
import csv
import unicodedata

# Make a copy
tmdb = movies.copy()

# Function to remove accents and quotes
def clean_text(v):
    if not isinstance(v, str):
        return v

    # Remove accents (é -> e, ü -> u, etc.)
    v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")

    # Remove double quotes
    v = v.replace('"', '')

    # Remove carriage returns/newlines inside fields
    v = v.replace("\r", " ").replace("\n", " ")

    return v.strip()

# Apply to every cell
tmdb = tmdb.map(clean_text)


# Export
tmdb.to_csv(
    "output/tmdb.csv",
    sep=",",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL
)

print("tmdb.csv:", tmdb.shape)

tmdb.csv: (1451806, 24)


**Oscars Data set from full_dat - winners and nominees**

In [11]:
oscars = pd.read_csv("full_data.csv", sep="\t")
print(oscars.shape)
oscars.head(3)



(12137, 14)


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins|The Patent Leather Kid,NaN,NaN
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexand...,NaN,NaN
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN


In [12]:
import csv
import unicodedata

# Make a copy
oscars_2 = oscars.copy()

# Function to remove accents and quotes
def clean_text(v):
    if not isinstance(v, str):
        return v

    # Remove accents (é -> e, ü -> u, etc.)
    v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")

    # Remove double quotes
    v = v.replace('"', '')

    # Remove carriage returns/newlines inside fields
    v = v.replace("\r", " ").replace("\n", " ")

    return v.strip()

# Apply to every cell
oscars_2 = oscars_2.map(clean_text)

# Drop Film column
oscars_2 = oscars_2.drop(columns=["Film"])

# Export
oscars_2.to_csv(
    "output/oscars_2.csv",
    sep=",",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL
)

print("oscars_db.csv:", oscars_2.shape)

oscars_db.csv: (12137, 13)
